# TPC$_{RP}$ Active Learning — CIFAR-10

Implementation of the **TypiClust (TPC$_{RP}$)** algorithm from:
> Hacohen, G., Dekel, A., & Weinshall, D. (2022). *Active Learning on a Budget: Opposite Strategies Suit High and Low Budgets.* ICML 2022.

**Strategy:** Self-supervised representation learning (simulated via pre-trained ResNet-18)  
→ K-Means clustering for diversity  
→ Typicality-based selection within each cluster.

In [ ]:
# ── Cell 1: Setup & Imports ──────────────────────────────────────────────────

# Mount Google Drive and add src/ to path when running on Colab
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/MachineLearning-Coursework2'
    sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))
else:
    # Local: repo root is one level above notebooks/
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))

# Standard library
import random
import warnings
warnings.filterwarnings('ignore')

# Numerical & ML
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

# Clustering & neighbours
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors

# Visualisation
import matplotlib
import matplotlib.pyplot as plt

# Project modules
from data_pipeline    import ActiveLearningDataset
from feature_extractor import ResNet18FeatureExtractor, extract_embeddings, build_extractor

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = (
    torch.device('cuda')  if torch.cuda.is_available() else
    torch.device('mps')   if torch.backends.mps.is_available() else
    torch.device('cpu')
)
print(f'Using device: {DEVICE}')
print(f'PyTorch {torch.__version__} | Torchvision {torchvision.__version__}')